#### Compute Keyframes and Bilateral Filtering  - (Single Video)

In [ ]:
import cv2
import numpy as np
import os

# Function to compute keyframes using optical flow
def compute_keyframes(video_path, output_folder, threshold=10):
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    
    keyframes = []
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Calculate optical flow
        flow = cv2.calcOpticalFlowFarneback(prev_gray, frame_gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        
        # Compute motion magnitude
        magnitude = np.linalg.norm(flow, axis=-1)
        
        # Check if frame is a keyframe
        if np.mean(magnitude) > threshold:
            # Apply bilateral filtering
            filtered_frame = cv2.bilateralFilter(frame, 9, 75, 75)
            keyframes.append(filtered_frame)
            
            keyframe_path = os.path.join(output_folder, f'kf_{frame_count}.jpg')
            cv2.imwrite(keyframe_path, filtered_frame)
            frame_count += 1
        
        prev_gray = frame_gray

    cap.release()
    return keyframes

if __name__ == '__main__':
    video_path = '1.mp4'
    output_folder = 'with_keyframes_filtered_images'
    os.makedirs(output_folder, exist_ok=True)
    
    keyframes = compute_keyframes(video_path, output_folder, threshold=10)
    
    print(f"Total Keyframes: {len(keyframes)}")

#### Applying on Multiple Videos

In [ ]:
import os
import cv2
import numpy as np
from glob import glob

# Function to apply bilateral filtering to an image
def apply_bilateral_filter(image):
    filtered_image = cv2.bilateralFilter(image, 9, 75, 75)
    return filtered_image

# Function to compute keyframes using optical flow and save filtered keyframes
def compute_and_save_keyframes(video_path, output_folder, threshold=10):
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

    keyframe_idx = 0
    keyframes = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Calculate optical flow
        flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)

        # Compute motion magnitude
        magnitude = np.linalg.norm(flow, axis=-1)

        # Check if frame is a keyframe
        if np.mean(magnitude) > threshold:
            keyframes.append(frame)
            filtered_frame = apply_bilateral_filter(frame)
            output_path = os.path.join(output_folder, f'filtered_keyframe_{keyframe_idx:04d}.jpg')
            cv2.imwrite(output_path, filtered_frame)
            keyframe_idx += 1

        prev_gray = gray

    cap.release()
    return keyframes

if __name__ == '__main__':
    video_folder = 'vid'   # Folder containing videos
    output_folder = 'multipe_videos_filtered_keyframes'   # Output folder for filtered keyframes
    os.makedirs(output_folder, exist_ok=True)

    video_files = glob(os.path.join(video_folder, '*.mp4'))
    for video_path in video_files:
        keyframes = compute_and_save_keyframes(video_path, output_folder, threshold=10)
        print(f"Video: {os.path.basename(video_path)}, Total Keyframes: {len(keyframes)}")

In [ ]:
# Applying on other videos dataset

import os
import cv2
import numpy as np
from glob import glob

# Function to apply bilateral filtering to an image
def apply_bilateral_filter(image):
    filtered_image = cv2.bilateralFilter(image, 9, 75, 75)
    return filtered_image

# Function to compute keyframes using optical flow and save filtered keyframes
def compute_and_save_keyframes(video_path, output_folder, threshold=10):
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

    keyframe_idx = 0
    keyframes = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Calculate optical flow
        flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)

        # Compute motion magnitude
        magnitude = np.linalg.norm(flow, axis=-1)

        # Check if frame is a keyframe
        if np.mean(magnitude) > threshold:
            keyframes.append(frame)
            filtered_frame = apply_bilateral_filter(frame)
            output_path = os.path.join(output_folder, f'filtered_keyframe_{keyframe_idx:04d}.jpg')
            cv2.imwrite(output_path, filtered_frame)
            keyframe_idx += 1

        prev_gray = gray

    cap.release()
    return keyframes

if __name__ == '__main__':
    video_folder = 'close-view-videos'   # Folder containing videos
    output_folder = 'close-view-multipe_videos_filtered_keyframes'   # Output folder for filtered keyframes
    os.makedirs(output_folder, exist_ok=True)

    video_files = glob(os.path.join(video_folder, '*.mkv'))
    for video_path in video_files:
        keyframes = compute_and_save_keyframes(video_path, output_folder, threshold=10)
        print(f"Video: {os.path.basename(video_path)}, Total Keyframes: {len(keyframes)}")

#### Applying Canny Edge Detection Approach

In [ ]:
import cv2
import os
import numpy as np

# Function to apply bilateral filtering to an image
def apply_bilateral_filter(image):
    filtered_image = cv2.bilateralFilter(image, 9, 75, 75)
    return filtered_image

# Function to compute keyframes using edge detection and feature matching
def compute_keyframes(video_path, output_folder, threshold=0.2):
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    prev_edges = cv2.Canny(prev_gray, threshold1=100, threshold2=200)

    keyframes = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        edges = cv2.Canny(gray, threshold1=100, threshold2=200)

        # Calculate the similarity score between current edges and previous edges
        similarity = np.mean(cv2.matchTemplate(edges, prev_edges, cv2.TM_CCOEFF_NORMED))

        if similarity < threshold:
            keyframes.append(frame)

        prev_edges = edges

    cap.release()

    return keyframes

if __name__ == '__main__':
    video_folder = 'videos'   # Folder containing video files
    output_folder = 'edge_matching_keyframes'   # Output folder for keyframes
    os.makedirs(output_folder, exist_ok=True)

    videos = os.listdir(video_folder)
    for video in videos:
        video_path = os.path.join(video_folder, video)
        keyframes = compute_keyframes(video_path, output_folder, threshold=0.2)

        # Apply bilateral filtering to keyframes and save them
        for i, keyframe in enumerate(keyframes):
            filtered_frame = apply_bilateral_filter(keyframe)
            output_path = os.path.join(output_folder, f'keyframe_{i}.jpg')
            cv2.imwrite(output_path, filtered_frame)

        print(f"Keyframes for {video} saved in {output_folder}")


#### Compute Keyframes and Bilateral Filtering with Motion Vectors using GMM - (Single Video)
--- Gaussian mixture models for background subtraction along with bilateral filtering to compute keyframes

In [ ]:
import cv2
import numpy as np
import os

# Function to apply bilateral filtering to an image
def apply_bilateral_filter(image):
    filtered_image = cv2.bilateralFilter(image, 9, 75, 75)
    return filtered_image

# Function to compute keyframes using motion detection and bilateral filtering
def compute_keyframes(video_path, output_folder, threshold_motion=1000, threshold_bilateral=10):
    cap = cv2.VideoCapture(video_path)
    bg_subtractor = cv2.createBackgroundSubtractorMOG2()

    keyframes = []
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Apply bilateral filtering
        filtered_frame = apply_bilateral_filter(frame)
        
        # Apply background subtraction
        fg_mask = bg_subtractor.apply(filtered_frame)
        
        # Compute motion magnitude
        motion_magnitude = np.sum(fg_mask)
        
        # Check if frame has significant motion and non-blurry content
        if motion_magnitude > threshold_motion and np.mean(fg_mask) > threshold_bilateral:
            keyframes.append((frame_count, filtered_frame))
            
            keyframe_path = os.path.join(output_folder, f'kf_{frame_count}.jpg')
            cv2.imwrite(keyframe_path, filtered_frame)
            frame_count += 1

    cap.release()
    return keyframes

if __name__ == '__main__':
    video_path = '1.mp4'
    output_folder = 'gmm_filtered_keyframes'
    os.makedirs(output_folder, exist_ok=True)
    
    keyframes = compute_keyframes(video_path, output_folder, threshold_motion=1000, threshold_bilateral=10)
    
    print(f"Total Keyframes: {len(keyframes)}")

#### Compute Keyframes and Bilateral Filtering with Motion Vectors using Absolute Differencing - (Single Video)

In [ ]:
import cv2
import numpy as np
import os

# Function to apply bilateral filtering to an image
def apply_bilateral_filter(image):
    filtered_image = cv2.bilateralFilter(image, 9, 75, 75)
    return filtered_image

# Function to compute keyframes using motion vectors and bilateral filtering
def compute_keyframes(video_path, output_folder, threshold_motion=1000, threshold_bilateral=10):
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()
    
    keyframes = []
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Apply bilateral filtering
        filtered_frame = apply_bilateral_filter(frame)
        
        # Compute absolute difference between frames
        frame_diff = cv2.absdiff(prev_frame, frame)
        
        # Compute motion magnitude
        motion_magnitude = np.sum(frame_diff)
        
        # Check if frame has significant motion and non-blurry content
        if motion_magnitude > threshold_motion and np.mean(frame_diff) > threshold_bilateral:
            keyframes.append((frame_count, filtered_frame))
            
            keyframe_path = os.path.join(output_folder, f'kf_{frame_count}.jpg')
            cv2.imwrite(keyframe_path, filtered_frame)
            frame_count += 1
        
        prev_frame = frame

    cap.release()
    return keyframes

if __name__ == '__main__':
    video_path = '1.mp4'
    output_folder = 'abd_filtered_keyframes'
    os.makedirs(output_folder, exist_ok=True)
    
    keyframes = compute_keyframes(video_path, output_folder, threshold_motion=1000, threshold_bilateral=10)
    
    print(f"Total Keyframes: {len(keyframes)}")

#### Compute Bilateral Filtering using Motion Vectors - (without computing keyframes)

In [ ]:
import cv2
import numpy as np
import os

# Function to apply bilateral filtering to an image
def apply_bilateral_filter(image):
    filtered_image = cv2.bilateralFilter(image, 9, 75, 75)
    return filtered_image

# Function to compute motion vectors using Gaussian Mixture Model
def compute_motion_vectors(video_path, output_folder, threshold=10):
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    
    motion_vectors = []
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Compute absolute difference between frames
        diff = cv2.absdiff(prev_gray, frame_gray)
        
        # Apply Gaussian Mixture Model to detect motion
        gmm = cv2.createBackgroundSubtractorMOG2()
        fg_mask = gmm.apply(diff)
        
        # Compute motion magnitude
        magnitude = np.linalg.norm(fg_mask, axis=-1)
        
        # Check if frame has motion
        if np.mean(magnitude) > threshold:
            # Apply bilateral filtering
            filtered_frame = apply_bilateral_filter(frame)
            motion_vectors.append((filtered_frame, magnitude))
            
            output_path = os.path.join(output_folder, f'filtered_frame_{frame_count}.jpg')
            cv2.imwrite(output_path, filtered_frame)
            frame_count += 1
        
        prev_gray = frame_gray

    cap.release()
    return motion_vectors

if __name__ == '__main__':
    video_path = '1.mp4'
    output_folder = 'without_keyframes_filtered_frames_using_motion_vectors_values'
    os.makedirs(output_folder, exist_ok=True)
    
    motion_vectors = compute_motion_vectors(video_path, output_folder, threshold=10)
    
    print(f"Total Filtered Frames with Motion: {len(motion_vectors)}")

    # Save motion vectors to a text file
    with open(os.path.join(output_folder, 'motion_vectors.txt'), 'w') as f:
        for idx, (_, magnitude) in enumerate(motion_vectors):
            f.write(f"Frame {idx}: {magnitude}\n")

#### Compute X-axis and Y-axis values instead of computing magnitude in above script

In [ ]:
import cv2
import numpy as np
import os

# Function to apply bilateral filtering to an image
def apply_bilateral_filter(image):
    filtered_image = cv2.bilateralFilter(image, 9, 75, 75)
    return filtered_image

# Function to compute motion vectors using Gaussian Mixture Model
def compute_motion_vectors(video_path, output_folder, threshold=10):
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    
    motion_vectors = []
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Compute absolute difference between frames
        diff = cv2.absdiff(prev_gray, frame_gray)
        
        # Apply Gaussian Mixture Model to detect motion
        gmm = cv2.createBackgroundSubtractorMOG2()
        fg_mask = gmm.apply(diff)
        
        # Compute x and y components of motion vectors
        motion_x = cv2.convertScaleAbs(fg_mask[..., 0])
        motion_y = cv2.convertScaleAbs(fg_mask[..., 1])
        
        # Check if frame has motion
        if np.mean(motion_x) > threshold or np.mean(motion_y) > threshold:
            # Apply bilateral filtering
            filtered_frame = apply_bilateral_filter(frame)
            motion_vectors.append((filtered_frame, motion_x, motion_y))
            
            output_path = os.path.join(output_folder, f'filtered_frame_{frame_count}.jpg')
            cv2.imwrite(output_path, filtered_frame)
            frame_count += 1
        
        prev_gray = frame_gray

    cap.release()
    return motion_vectors

if __name__ == '__main__':
    video_path = '1.mp4'
    output_folder = 'without_keyframes_filtered_frames_using_motion_vectors_values_updated'
    os.makedirs(output_folder, exist_ok=True)
    
    motion_vectors = compute_motion_vectors(video_path, output_folder, threshold=10)
    
    print(f"Total Filtered Frames with Motion: {len(motion_vectors)}")

    # Save motion vectors to a text file
    with open(os.path.join(output_folder, 'updated_motion_vectors.txt'), 'w') as f:
        for idx, (_, motion_x, motion_y) in enumerate(motion_vectors):
            f.write(f"Frame {idx}: X-axis: {np.mean(motion_x)}, Y-axis: {np.mean(motion_y)}\n")

#### Pixel-by-Pixel Value

In [ ]:
# EXP-1

import cv2
import numpy as np
import os

# Function to apply bilateral filtering to an image
def apply_bilateral_filter(image):
    filtered_image = cv2.bilateralFilter(image, 9, 75, 75)
    return filtered_image

# Function to compute motion detection using Gaussian Mixture Model
def compute_motion_detection(video_path, output_folder, threshold=10):
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    
    motion_detections = []
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Compute absolute difference between frames
        diff = cv2.absdiff(prev_gray, frame_gray)
        
        # Apply Gaussian Mixture Model to detect motion
        gmm = cv2.createBackgroundSubtractorMOG2()
        fg_mask = gmm.apply(diff)
        
        # Check if frame has motion
        if np.mean(fg_mask) > threshold:
            # Apply bilateral filtering
            filtered_frame = apply_bilateral_filter(frame)
            motion_detections.append((filtered_frame, fg_mask))
            
            output_path = os.path.join(output_folder, f'filtered_frame_{frame_count}.jpg')
            cv2.imwrite(output_path, filtered_frame)
            frame_count += 1
        
        prev_gray = frame_gray

    cap.release()
    return motion_detections

if __name__ == '__main__':
    video_path = '1.mp4'
    output_folder = 'motion_detections'
    os.makedirs(output_folder, exist_ok=True)
    
    motion_detections = compute_motion_detection(video_path, output_folder, threshold=10)
    
    print(f"Total Frames with Motion Detected: {len(motion_detections)}")

    # Save motion detection masks to a text file
    with open(os.path.join(output_folder, 'motion_detections.txt'), 'w') as f:
        for idx, (_, motion_mask) in enumerate(motion_detections):
            f.write(f"Frame {idx}:\n")
            f.write(f"{motion_mask}\n")

In [ ]:
import cv2
import numpy as np
import os

# Function to apply bilateral filtering to an image
def apply_bilateral_filter(image):
    filtered_image = cv2.bilateralFilter(image, 9, 75, 75)
    return filtered_image

# Function to compute motion vectors using Gaussian Mixture Model
def compute_motion_vectors(video_path, output_folder, threshold=10):
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Compute absolute difference between frames
        diff = cv2.absdiff(prev_gray, frame_gray)
        
        # Apply Gaussian Mixture Model to detect motion
        gmm = cv2.createBackgroundSubtractorMOG2()
        fg_mask = gmm.apply(diff)
        
        # Compute motion magnitude
        magnitude = np.linalg.norm(fg_mask, axis=-1)
        
        # Check if frame has motion
        if np.mean(magnitude) > threshold:
            # Apply bilateral filtering
            filtered_frame = apply_bilateral_filter(frame)
            
            # Save motion vectors as individual image files
            motion_vectors_path = os.path.join(output_folder, f'motion_vectors_frame_{frame_count}.jpg')
            cv2.imwrite(motion_vectors_path, fg_mask)
            
            filtered_frame_path = os.path.join(output_folder, f'filtered_frame_{frame_count}.jpg')
            cv2.imwrite(filtered_frame_path, filtered_frame)
            
            frame_count += 1
        
        prev_gray = frame_gray

    cap.release()

if __name__ == '__main__':
    video_path = '1.mp4'
    output_folder = 'filtered_frames_with_motion_vectors_idividually'
    os.makedirs(output_folder, exist_ok=True)
    
    compute_motion_vectors(video_path, output_folder, threshold=10)

### Final File Aug 22, 2023

In [ ]:
import cv2
import numpy as np
import os

# Function to apply bilateral filtering to an image
def apply_bilateral_filter(image):
    filtered_image = cv2.bilateralFilter(image, 9, 75, 75)
    return filtered_image

# Function to compute motion vectors using Gaussian Mixture Model
def compute_motion_vectors(video_path, output_folder_frames, output_folder_vectors, threshold=10):
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Compute absolute difference between frames
        diff = cv2.absdiff(prev_gray, frame_gray)
        
        # Apply Gaussian Mixture Model to detect motion
        gmm = cv2.createBackgroundSubtractorMOG2()
        fg_mask = gmm.apply(diff)
        
        # Compute motion magnitude
        magnitude = np.linalg.norm(fg_mask, axis=-1)
        
        # Check if frame has motion
        if np.mean(magnitude) > threshold:
            # Apply bilateral filtering
            filtered_frame = apply_bilateral_filter(frame)
            
            # Save image frame
            frame_path = os.path.join(output_folder_frames, f'frame_{frame_count}.jpg')
            cv2.imwrite(frame_path, filtered_frame)
            
            # Save motion vectors as text file
            motion_vectors_path = os.path.join(output_folder_vectors, f'mv_frame_{frame_count}.txt')
            np.savetxt(motion_vectors_path, fg_mask, fmt='%d', delimiter=',')
            
            frame_count += 1
        
        prev_gray = frame_gray

    cap.release()

if __name__ == '__main__':
    video_path = '2.mp4'
    output_folder_frames = 'frames_aug_28'
    output_folder_vectors = 'mv_aug_28'
    os.makedirs(output_folder_frames, exist_ok=True)
    os.makedirs(output_folder_vectors, exist_ok=True)
    
    compute_motion_vectors(video_path, output_folder_frames, output_folder_vectors, threshold=10)


In [ ]:
import cv2
import numpy as np
import os

# Function to apply bilateral filtering to an image
def apply_bilateral_filter(image):
    filtered_image = cv2.bilateralFilter(image, 9, 75, 75)
    return filtered_image

# Function to compute motion vectors using dense optical flow
def compute_motion_vectors(video_path, output_folder_frames, output_folder_vectors, threshold=10):
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Compute dense optical flow
        flow = cv2.calcOpticalFlowFarneback(prev_gray, frame_gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        
        # Compute motion magnitude
        magnitude = np.linalg.norm(flow, axis=-1)
        
        # Check if frame has motion
        if np.mean(magnitude) > threshold:
            # Apply bilateral filtering
            filtered_frame = apply_bilateral_filter(frame)
            
            # Save image frame
            frame_path = os.path.join(output_folder_frames, f'frame_{frame_count}.jpg')
            cv2.imwrite(frame_path, filtered_frame)
            
            # Save motion vectors as text file
            motion_vectors_path = os.path.join(output_folder_vectors, f'mv_frame_{frame_count}.txt')
            np.savetxt(motion_vectors_path, flow.reshape(-1, 2), fmt='%.2f', delimiter=',')
            
            frame_count += 1
        
        prev_gray = frame_gray

    cap.release()

if __name__ == '__main__':
    video_path = '1.mp4'
    output_folder_frames = 'frames_aug_28'
    output_folder_vectors = 'mv_aug_28'
    os.makedirs(output_folder_frames, exist_ok=True)
    os.makedirs(output_folder_vectors, exist_ok=True)
    
    compute_motion_vectors(video_path, output_folder_frames, output_folder_vectors, threshold=10)

In [ ]:
with open('mv_aug_29/mv_frame_0_x.txt','rt') as stream:
    data = stream.read()
L = data.split('\n')
print(len(L))
K = [x for n in range(len(L)-1) for x in eval(L[n]) ]
print(len(K),1920*1080)

In [ ]:
with open('mv_aug_29/mv_frame_0_y.txt','rt') as stream:
    data = stream.read()
L = data.split('\n')
print(len(L))
K = [x for n in range(len(L)-1) for x in eval(L[n]) ]
print(len(K),1920*1080)

### Shape of Motion Vectors in Tensors:

In [ ]:
import cv2
import numpy as np
import os

# Function to apply bilateral filtering to an image
def apply_bilateral_filter(image):
    filtered_image = cv2.bilateralFilter(image, 9, 75, 75)
    return filtered_image

# Function to compute motion vectors using dense optical flow
def compute_motion_vectors(video_path, output_folder_frames, output_folder_vectors, threshold=10):
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Compute dense optical flow
        flow = cv2.calcOpticalFlowFarneback(prev_gray, frame_gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        
        print("Shape of motion vectors:", flow.shape)  # Print the shape of the motion vectors tensor
        
        # Compute motion magnitude
        magnitude = np.linalg.norm(flow, axis=-1)
        
        # Check if frame has motion
        if np.mean(magnitude) > threshold:
            # Apply bilateral filtering
            filtered_frame = apply_bilateral_filter(frame)
            
            # Save image frame
            frame_path = os.path.join(output_folder_frames, f'frame_{frame_count}.jpg')
            cv2.imwrite(frame_path, filtered_frame)
            
            # Save motion vectors as a single text file
            motion_vectors_path = os.path.join(output_folder_vectors, f'mv_frame_{frame_count}.txt')
            np.savetxt(motion_vectors_path, flow.reshape(-1, 2), fmt='%.2f', delimiter=',')
            
            frame_count += 1
        
        prev_gray = frame_gray

    cap.release()

if __name__ == '__main__':
    video_path = '1.mp4'
    output_folder_frames = 'frames_aug_v5'
    output_folder_vectors = 'mv_aug_v5'
    os.makedirs(output_folder_frames, exist_ok=True)
    os.makedirs(output_folder_vectors, exist_ok=True)
    
    compute_motion_vectors(video_path, output_folder_frames, output_folder_vectors, threshold=10)

In [ ]:
with open('mv_aug_30/mv_frame_1.txt','rt') as stream:
    data = stream.read()
L = data.split('\n')
print(len(L))
K = [x for n in range(len(L)-1) for x in eval(L[n]) ]
print(len(K),1920*1080)

#### Individual x and Y components storing in .txt file

In [1]:
import cv2
import numpy as np
import os

# Function to apply bilateral filtering to an image
def apply_bilateral_filter(image):
    filtered_image = cv2.bilateralFilter(image, 9, 75, 75)
    return filtered_image

# Function to compute motion vectors using dense optical flow
def compute_motion_vectors(video_path, output_folder_frames, output_folder_vectors, threshold=10):
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Compute dense optical flow
        flow = cv2.calcOpticalFlowFarneback(prev_gray, frame_gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        
        # Compute motion magnitude
        magnitude = np.linalg.norm(flow, axis=-1)
        
        # Check if frame has motion
        if np.mean(magnitude) > threshold:
            # Apply bilateral filtering
            filtered_frame = apply_bilateral_filter(frame)
            
            # Save image frame
            frame_path = os.path.join(output_folder_frames, f'frame_{frame_count}.jpg')
            cv2.imwrite(frame_path, filtered_frame)
            
            # Save motion vectors as text files (separate x and y components)
            motion_vectors_path_x = os.path.join(output_folder_vectors, f'mv_frame_{frame_count}_x.txt')
            motion_vectors_path_y = os.path.join(output_folder_vectors, f'mv_frame_{frame_count}_y.txt')
            np.savetxt(motion_vectors_path_x, flow[:, :, 0], fmt='%.2f', delimiter=',')
            np.savetxt(motion_vectors_path_y, flow[:, :, 1], fmt='%.2f', delimiter=',')
            
            frame_count += 1
        
        prev_gray = frame_gray

    cap.release()

if __name__ == '__main__':
    video_path = '1.mp4'
    output_folder_frames = 'frames_aug_v15'
    output_folder_vectors = 'mv_aug_15'
    os.makedirs(output_folder_frames, exist_ok=True)
    os.makedirs(output_folder_vectors, exist_ok=True)
    
    compute_motion_vectors(video_path, output_folder_frames, output_folder_vectors, threshold=10)

#### Aug 29 Updates   - Optical Flow (x and y storing in single .txt file)

In [2]:
import cv2
import numpy as np
import os

# Function to apply bilateral filtering to an image
def apply_bilateral_filter(image):
    filtered_image = cv2.bilateralFilter(image, 9, 75, 75)
    return filtered_image

def compute_motion_vectors(video_path, output_folder_frames, output_folder_vectors, threshold=10):
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Compute optical flow between frames
        flow = cv2.calcOpticalFlowFarneback(prev_gray, frame_gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        
        # Calculate individual motion vectors (xyxy format)
        motion_vectors = flow.reshape(-1, 2)
        
        # Calculate magnitude to decide if there's motion
        magnitude = np.linalg.norm(motion_vectors, axis=1)
        
        # Check if frame has motion
        if np.mean(magnitude) > threshold:
            # Apply bilateral filtering
            filtered_frame = apply_bilateral_filter(frame)
            
            # Save image frame
            frame_path = os.path.join(output_folder_frames, f'frame_{frame_count}.jpg')
            cv2.imwrite(frame_path, filtered_frame)
            
            # Save motion vectors as text file (in "xyxy" format)
            motion_vectors_path = os.path.join(output_folder_vectors, f'mv_frame_{frame_count}.txt')
            np.savetxt(motion_vectors_path, motion_vectors, fmt='%.3f', delimiter=',')
            
            frame_count += 1
        
        prev_gray = frame_gray

    cap.release()
if __name__ == '__main__':
    video_path = 'v1.mp4'
    output_folder_frames = 'frames_v7'
    output_folder_vectors = 'mv_v7'
    os.makedirs(output_folder_frames, exist_ok=True)
    os.makedirs(output_folder_vectors, exist_ok=True)
    
    compute_motion_vectors(video_path, output_folder_frames, output_folder_vectors, threshold=10)

### Images

In [1]:
import cv2
import numpy as np
import os

# Function to apply bilateral filtering to an image
def apply_bilateral_filter(image):
    filtered_image = cv2.bilateralFilter(image, 9, 75, 75)
    return filtered_image

# Function to compute motion vectors using dense optical flow
def compute_motion_vectors(image_folder, output_folder_vectors, threshold=10):
    image_paths = sorted([os.path.join(image_folder, img) for img in os.listdir(image_folder) if img.endswith('.jpg') or img.endswith('.png')])
    
    motion_vectors = []

    prev_frame = cv2.imread(image_paths[0])
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

    for image_path in image_paths[1:]:
        frame = cv2.imread(image_path)
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Apply bilateral filtering to the frame
        filtered_frame = apply_bilateral_filter(frame)
        
        # Compute dense optical flow
        flow = cv2.calcOpticalFlowFarneback(prev_gray, frame_gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        
        # Calculate individual motion vectors (xyxy format)
        motion_vectors.append(flow.reshape(-1, 2))
        
        prev_gray = frame_gray

    # Convert the list of motion vectors to a NumPy array
    motion_vectors = np.array(motion_vectors)
    
    # Save motion vectors as text file (in "xyxy" format)
    motion_vectors_path = os.path.join(output_folder_vectors, 'mv_seq_0.txt')
    np.savetxt(motion_vectors_path, motion_vectors.reshape(-1, 2), fmt='%.3f', delimiter=',')

if __name__ == '__main__':
    image_folder = 'seq0'  # Folder containing the input images
    output_folder_vectors = 'mv_seq0'
    os.makedirs(output_folder_vectors, exist_ok=True)
    
    compute_motion_vectors(image_folder, output_folder_vectors, threshold=10)

In [ ]:
## load motion vectors

import numpy as np
with open('./ssd1/motion_vectors/new2/mv_combined.txt','rt') as stream:
    data = stream.read()
L = data.replace('\n',',')
K = eval(L)
print(len(K))
print(K[:100])
mvs = np.array(K).reshape(5,1080,1920,2)

In [ ]:
# plots

import cv2
import matplotlib.pyplot as plt
from glob import glob
from PIL import Image

images = glob('./ssd1/motion_vectors/new2/*.png')

def fmod(x,y):
    return x-np.floor(x/y)*y

def Sobel3(I):
    R = I.astype('float32')
    Gx = R[:-2,:-2,:]-R[2:,:-2,:]+2*(R[:-2,1:-1,:]-R[2:,1:-1,:])+R[:-2,2:,:]-R[2:,2:,:]
    Gy = R[:-2,:-2,:]-R[:-2,2:,:]+2*(R[1:-1,:-2,:]-R[1:-1,2:,:])+R[2:,:-2,:]-R[2:,2:,:]
    R = np.sqrt(np.sum(Gx*2,axis=2)+np.sum(Gy*2,axis=2))
    return np.pad(R/np.max(R),((1,1),(1,1)),'symmetric')

for n,i in enumerate(images):
    img = np.array(Image.open(i))
    mv = mvs[n]
    # Use Hue, Saturation, Value colour model 
    hsv = np.zeros((mv.shape[0],mv.shape[1],3), dtype='float32')
    hsv[... ,1] = 0.5
    mag = np.sqrt(mv[..., 0]*2+mv[..., 1]*2)
    ang = np.arctan2(mv[...,1],mv[...,0])
    hsv[..., 0] = fmod(ang / (2*np.pi) * 255,255)
    hsv[..., 2] = 0+1*mag/np.max(mag)
    rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
    Srgb = Sobel3(mv)
    #rgb[Srgb>0.2]=0
    Simg = Sobel3(img)
    fig, ax = plt.subplots(2,2,figsize=(12.8*2,7.2*2))
    ax[0,0].imshow(img)
    ax[0,0].axis('off')
    ax[0,1].imshow(1-Simg,cmap='gray')
    ax[0,1].axis('off')
    ax[1,0].imshow(rgb)
    ax[1,0].axis('off')
    ax[1,1].imshow(1-Srgb,cmap='gray')
    ax[1,1].axis('off')
    plt.show()